In [1]:
import duckdb
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo
import os

In [2]:
WAREHOUSE_TEST = "./test_duckdb_data/gtftest23.duckdb"

con = duckdb.connect(WAREHOUSE_TEST)
con.execute("SET TimeZone='UTC';")

con.sql("SHOW TABLES").df()

,name
0,agency
1,calendar
2,calendar_dates
3,delays_with_support_columns
4,feed_info
5,routes
6,shapes
7,stop_times
8,stops
9,trips


In [3]:
df = con.sql("""
    SELECT
        s.stop_id,
        s.stop_name,
        s.stop_lat,
        s.stop_lon,
        ROUND(AVG(pe.delay_min), 2) AS avg_delay_min
    FROM stops s
    LEFT JOIN delays_with_support_columns pe
        ON s.stop_id = pe.stop_id
    GROUP BY s.stop_id, s.stop_name, s.stop_lat, s.stop_lon
    ORDER BY avg_delay_min DESC
""").df()

df

,stop_id,stop_name,stop_lat,stop_lon,avg_delay_min
0,2624,Cassin / Kirchner,43.672277,7.224127,1.27
1,2622,Ferber,43.676896,7.228362,1.27
2,2771,Parc Phoenix,43.669416,7.219098,1.27
3,2767,Grand Arénas,43.669944,7.212342,1.27
4,1173,Route d'Aspremont,43.749012,7.247345,1.13
...,...,...,...,...,...
4480,7316,Georges Bonjean,43.800037,7.273633,NaN
4481,place_PKSLS,Parking Salèse,44.126003,7.260799,NaN
4482,23021,Parking Salèse,44.126003,7.260799,NaN
4483,place_SMGEN,Gendarmerie,44.078155,7.24978,NaN


In [4]:
EXPORT_DIR = "./exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

def current_timestamp_string():
    return datetime.now(ZoneInfo("Europe/Paris")).strftime("%Y%m%d_%H%M%S")

csv_path = f"{EXPORT_DIR}/avg_delay_by_stop_{current_timestamp_string()}.csv"
df.to_csv(csv_path, index=False)
csv_path

'./exports/avg_delay_by_stop_20250912_105120.csv'

# Question 4 lignes les plus en retard

In [5]:
con.sql("SHOW TABLES").df()

,name
0,agency
1,calendar
2,calendar_dates
3,delays_with_support_columns
4,feed_info
5,routes
6,shapes
7,stop_times
8,stops
9,trips


In [6]:
con.sql("SELECT * FROM trips LIMIT 5").df()

,route_id,service_id,trip_id,trip_headsign,trip_short_name,direction_id,shape_id,wheelchair_accessible,bikes_allowed
0,75,RESEAU2023-75-Dimanche-69-75,4962940-75_R_97_7501_11:50-RESEAU2023-75-Diman...,Gare SNCF,4962940-75_R_97_7501_11:50-RESEAU2023-75-Diman...,1,750032,0,0
1,75,RESEAU2023-75-Samedi-77-75,4962972-75_R_97_7501_17:45-RESEAU2023-75-Samed...,Gare SNCF,4962972-75_R_97_7501_17:45-RESEAU2023-75-Samed...,1,750032,0,0
2,40,RESEAU2023-40-Semaine-04-40,4350596-40_A_50_4001_14:53-RESEAU2023-40-Semai...,Collège Matisse,4350596-40_A_50_4001_14:53-RESEAU2023-40-Semai...,0,400009,0,0
3,62,RESEAU2023-62-Samedi-10-62,4336738-62_R_98_6203_18:10-RESEAU2023-62-Samed...,Magnan,4336738-62_R_98_6203_18:10-RESEAU2023-62-Samed...,1,620015,0,0
4,C34,RESEAU2021-C34-Dimanche-02-C34,3064040-C34_R_2_C3401_09:45-RESEAU2021-C34-Dim...,None,3064040-C34_R_2_C3401_09:45-RESEAU2021-C34-Dim...,1,C340002,0,0


In [8]:
df = con.sql("""
    SELECT
        t.route_id,
        ROUND(AVG(d.delay_min), 2) AS avg_delay_min,
        COUNT(*) AS num_events
    FROM delays_with_support_columns d
    LEFT JOIN trips t
        ON d.trip_id = t.trip_id
    GROUP BY t.route_id
    ORDER BY avg_delay_min DESC
""").df()

df

,route_id,avg_delay_min,num_events
0,B,0.54,10
1,L2,0.34,33
2,15,0.22,17
3,63,0.15,25
4,51,0.00,1
5,22,0.00,1
6,60,-0.01,46
7,L1,-0.04,64
8,21,-0.04,10
9,32,-0.08,4


In [9]:
EXPORT_DIR = "./exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

def current_timestamp_string():
    return datetime.now(ZoneInfo("Europe/Paris")).strftime("%Y%m%d_%H%M%S")

csv_path = f"{EXPORT_DIR}/avg_delay_by_route_{current_timestamp_string()}.csv"
df.to_csv(csv_path, index=False)
csv_path

'./exports/avg_delay_by_route_20250912_115100.csv'